[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_03_Data_Preprocessing/01_data_preprocessing.ipynb)

# Episode 6 & 7 – Data Preprocessing

**Machine Learning Bootcamp** | Module 03

---

## 🎯 Learning Objectives
- Handle missing values and outliers
- Encode categorical variables
- Scale numerical features
- Split data into training and test sets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid')

## 1. Create a Messy Dataset

In [ ]:
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'age':      np.random.randint(18, 70, n).astype(float),
    'income':   np.random.normal(50000, 15000, n),
    'city':     np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n),
    'gender':   np.random.choice(['M', 'F'], n),
    'purchased': np.random.randint(0, 2, n)
})

# Introduce missing values
df.loc[np.random.choice(n, 20, replace=False), 'age']    = np.nan
df.loc[np.random.choice(n, 15, replace=False), 'income'] = np.nan

# Introduce outliers
df.loc[0, 'income'] = 500000
df.loc[1, 'income'] = -5000

print('Shape:', df.shape)
print('Missing values:\n', df.isnull().sum())
df.head()

## 2. Handling Missing Values

In [ ]:
# Strategy 1: Drop rows with any missing value (use sparingly!)
df_dropped = df.dropna()
print(f'After dropna: {df_dropped.shape[0]} rows (lost {n - df_dropped.shape[0]})')

# Strategy 2: Impute (fill) missing values
df_clean = df.copy()

# Numerical: fill with median
num_imputer = SimpleImputer(strategy='median')
df_clean[['age', 'income']] = num_imputer.fit_transform(df_clean[['age', 'income']])

print(f'\nMissing values after imputation:\n{df_clean.isnull().sum()}')

## 3. Handling Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df_clean['income'], ax=axes[0])
axes[0].set_title('Income – Before Outlier Removal')

# IQR method
Q1, Q3 = df_clean['income'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
df_clean = df_clean[(df_clean['income'] >= lower) & (df_clean['income'] <= upper)]

sns.boxplot(y=df_clean['income'], ax=axes[1])
axes[1].set_title('Income – After IQR Outlier Removal')
plt.tight_layout(); plt.show()
print(f'Rows remaining: {len(df_clean)}')

## 4. Encoding Categorical Variables

In [ ]:
# Label Encoding (for ordinal or binary categories)
le = LabelEncoder()
df_clean['gender_encoded'] = le.fit_transform(df_clean['gender'])
print('Label Encoding (gender):', df_clean[['gender', 'gender_encoded']].drop_duplicates().values)

# One-Hot Encoding (for nominal categories)
df_encoded = pd.get_dummies(df_clean, columns=['city'], drop_first=False)
print('\nNew columns after OHE:', [c for c in df_encoded.columns if c.startswith('city')])
df_encoded.head(3)

## 5. Feature Scaling

In [ ]:
features = ['age', 'income']

# StandardScaler: zero mean, unit variance
std_scaler = StandardScaler()
df_std = df_clean.copy()
df_std[features] = std_scaler.fit_transform(df_std[features])
print('StandardScaler – age mean/std:', df_std['age'].mean().round(5), df_std['age'].std().round(5))

# MinMaxScaler: [0, 1] range
mm_scaler = MinMaxScaler()
df_mm = df_clean.copy()
df_mm[features] = mm_scaler.fit_transform(df_mm[features])
print('MinMaxScaler  – age min/max: ', df_mm['age'].min().round(3), df_mm['age'].max().round(3))

## 6. Train / Test Split

In [ ]:
feature_cols = ['age', 'income', 'gender_encoded']
X = df_std[feature_cols]
y = df_std['purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'Class balance (train): {y_train.value_counts(normalize=True).round(2).to_dict()}')

## 🏋️ Exercises

1. Load the Titanic dataset and apply the full preprocessing pipeline: impute missing Age with median, encode Sex and Embarked, drop irrelevant columns.
2. What happens if you use `StandardScaler` on a feature with heavy outliers? Compare it with `RobustScaler`.
3. Try `IterativeImputer` (from `sklearn.impute`) for multivariate imputation and compare results with `SimpleImputer`.

---
**Next ▶ [Module 04 – Regression](../Module_04_Regression/01_linear_regression.ipynb)**